# EEC2204 전자기학 2 — 텀 프로젝트 시작 노트북(starter notebook)
## 경두개자기자극(Transcranial Magnetic Stimulation, TMS)의 전자기장 모델링

이 노트북은 `task1~3_*.py`에 구현한 함수를 불러와(import) 각 Task의 검증을 실행하고 보고서용 그림을 그리는 작업 공간임.

- 함수 구현은 노트북이 아니라 `task1_coil_field.py`, `task2_tissue_induction.py`, `task3_neuron.py`의 `# === TODO [Task X.Y] ===` 블록에서 함.
- 노트북은 `.py` 파일과 같은 폴더(`starter_code/`)에서 열어야 하며, `tms_config.py`와 `neuron_model.py`는 수정하지 않음.
- 첫 코드 셀의 `%autoreload 2`로 인해 `.py` 파일을 수정·저장하면 커널을 재시작하지 않아도 다음 셀 실행부터 반영됨.
- 셀은 위에서부터 순서대로 실행함. 함수의 인자(argument)와 입출력(input/output) 형식은 각 함수의 docstring을, 과제 설명은 저장소 루트의 `README.md`(과제 명세서)를 참고함.

| Task | 내용 | 구현 파일 | 배점 |
|:---:|---|---|:---:|
| Task 1 | 코일 자기장과 인덕턴스 | `task1_coil_field.py` | 30점 |
| Task 2 | 조직 레이어에서 유도 전기장 (Task 2.1은 보고서만) | `task2_tissue_induction.py` | 40점 |
| Task 3 | 대뇌 피질 뉴런 활성화 | `task3_neuron.py` | 20점 |
| Task 4 | 종합 고찰 (보고서만) | — | 10점 |


### 함수 불러오기(import)
`task1~3_*.py`에 구현한 함수와 `tms_config.py`의 매개변수를 불러옴.


In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np

from neuron_model import find_electric_field_threshold, simulate_hh_neuron
from tms_config import (
    DIDT_RISE,
    E_GOAL_TRADEOFF,
    H_NOMINAL,
    I_PEAK_NOMINAL,
    K_SEG,
    MU0,
    N_TURNS,
    R_CIRC,
    R_FIG8,
    R_OFF_TARGET,
    R_TARGET,
    S_FIG8,
    T_FALL_S,
    T_HOLD_S,
    T_MS,
    T_RISE_S,
    T_S,
    U_NEURON,
    WIRE_RADIUS,
    Z_TARGET,
    get_layer_properties,
)
from task1_coil_field import (
    make_circular_loop,
    build_circular_coil,
    build_figure8_coil,
    compute_unit_A_and_B,
    analytical_circular_on_axis_Bz,
    verify_curl_A,
    trapezoidal_pulse_current_and_didt,
    compute_inductance,
)
from task2_tissue_induction import (
    compute_depth_profile,
    compute_target_waveforms,
    compute_cortical_metrics,
    sweep_wing_radius,
    sweep_standoff,
    sweep_lateral_offset,
)
from task3_neuron import (
    compute_E_eff_waveform,
    compute_fig8_thresholds,
    find_max_standoff,
)


## Task 1 — 코일 자기장과 인덕턴스 (30점)


### Task 1.1 — 중심축 $B_z$ 해석해와 수치해 비교 (15점)
손으로 유도한 원형 코일의 중심축 $B_z$ 해석해와 비오-사바르(Biot–Savart) 수치해를 비교하여 코일 모델이 정확한지 검증함.

구현할 함수(`task1_coil_field.py`): `make_circular_loop`, `build_figure8_coil`, `compute_unit_A_and_B`, `analytical_circular_on_axis_Bz`, `verify_curl_A`


**검증 실행(check):** `task1_coil_field.py`에서 위 함수를 구현·저장한 뒤 아래 셀을 실행하면 검증 값이 출력됨.


In [ ]:
circ_pts, circ_dl = build_circular_coil()
fig8_pts, fig8_dl = build_figure8_coil()


# --- Task 1.1: 중심축 B_z 해석해 vs. 수치해 ---
z_axis = np.linspace(-60e-3, 0.0, 121)
obs_axis = np.column_stack([np.zeros_like(z_axis), np.zeros_like(z_axis), z_axis])
_, b_axis = compute_unit_A_and_B(circ_pts, circ_dl, obs_axis)
bz_num = b_axis[:, 2] * I_PEAK_NOMINAL
bz_ana = analytical_circular_on_axis_Bz(z_axis, current=I_PEAK_NOMINAL)
max_rel_err = float(np.max(np.abs(bz_num - bz_ana) / np.abs(bz_ana)))
print(f"[Task 1.1] On-axis Bz max rel error = {max_rel_err:.3e} (goal < 1e-4)")

# --- Task 1.1: B = ∇×A 검증 ---
_, _, err_c = verify_curl_A(circ_pts, circ_dl, R_OFF_TARGET)
_, _, err_f = verify_curl_A(fig8_pts, fig8_dl, R_TARGET)
print(f"[Task 1.1] curl(A) rel error: Circular @ (45,0,-20) mm = {err_c:.3e}, Figure-8 @ (0,0,-20) mm = {err_f:.3e}")


### Task 1.2 — 인덕턴스와 구동 전압 (15점)
벡터 퍼텐셜의 쇄교 자속(flux linkage)으로 코일 인덕턴스를 계산하고, 사다리꼴 전류 펄스를 만드는 데 필요한 구동 전압과 저장 에너지를 구함.

구현할 함수(`task1_coil_field.py`): `trapezoidal_pulse_current_and_didt`, `compute_inductance`


**검증 실행(check):** `task1_coil_field.py`에서 위 함수를 구현·저장한 뒤 아래 셀을 실행하면 검증 값이 출력됨.


In [ ]:
# --- Task 1.2: 인덕턴스 분해와 구동 전압 ---
i_t, _ = trapezoidal_pulse_current_and_didt(T_S)
i_peak = float(np.max(np.abs(i_t)))

wing1_pts, wing1_dl = make_circular_loop(-S_FIG8, 0.0, H_NOMINAL, R_FIG8, direction_sign=+1.0)
wing2_pts, wing2_dl = make_circular_loop(+S_FIG8, 0.0, H_NOMINAL, R_FIG8, direction_sign=-1.0)
l_circ = compute_inductance(circ_pts, circ_dl)
l_w1 = compute_inductance(wing1_pts, wing1_dl)
l_w2 = compute_inductance(wing2_pts, wing2_dl)
m_12 = compute_inductance(wing1_pts, wing1_dl, wing2_pts, wing2_dl)
l_fig8 = compute_inductance(fig8_pts, fig8_dl)

print(f"[Task 1.2] L_circ = {l_circ*1e6:.3f} uH | V_coil = +/-{l_circ*DIDT_RISE*1e-3:.3f} kV | Wm = {0.5*l_circ*i_peak**2:.2f} J")
print(f"           Figure-8: L1 = {l_w1*1e6:.3f} uH, L2 = {l_w2*1e6:.3f} uH, M12 = {m_12*1e6:+.3f} uH")
print(f"           L1 + L2 + 2*M12 = {(l_w1 + l_w2 + 2*m_12)*1e6:.6f} uH vs direct L_fig8 = {l_fig8*1e6:.6f} uH")
print(f"           V_coil = +/-{l_fig8*DIDT_RISE*1e-3:.3f} kV | Wm = {0.5*l_fig8*i_peak**2:.2f} J")


## Task 2 — 조직 레이어에서 유도 전기장 (40점)


### Task 2.1 — 유도 전기장이 전도도와 무관함을 증명 (12점, 보고서만)
코드 구현 없이 보고서로 제출함.


### Task 2.2 — 깊이 분포, 시간 파형, 피질 분포도 (14점)
조직 내 유도 전기장과 와전류(eddy current)를 계산하고, 두 코일의 초점성(focality)을 비교함.

구현할 함수(`task2_tissue_induction.py`): `compute_depth_profile`, `compute_target_waveforms`, `compute_cortical_metrics`


**검증 실행(check):** `task2_tissue_induction.py`에서 위 함수를 구현·저장한 뒤 아래 셀을 실행하면 검증 값이 출력됨.


In [ ]:
circ_pts, circ_dl = build_circular_coil()
fig8_pts, fig8_dl = build_figure8_coil()
l_circ = compute_inductance(circ_pts, circ_dl)
l_fig8 = compute_inductance(fig8_pts, fig8_dl)
i_t, di_dt = trapezoidal_pulse_current_and_didt(T_S)


# --- Task 2.2a: 핫스팟 아래 깊이 분포 ---
_, e_f8, j_f8 = compute_depth_profile(fig8_pts, fig8_dl, x_line=0.0)
_, e_cc, j_cc = compute_depth_profile(circ_pts, circ_dl, x_line=R_CIRC)
print(f"[Task 2.2a] Peak |J| (CSF shunting): Figure-8 = {np.max(j_f8):.2f} A/m^2, Circular = {np.max(j_cc):.2f} A/m^2")

# --- Task 2.2b: 표적점의 |B(t)| vs |E(t)| ---
b_t, e_t = compute_target_waveforms(fig8_pts, fig8_dl, R_TARGET, i_t, di_dt)
hold = (T_S >= 0.07e-3) & (T_S < 0.11e-3)
print(f"[Task 2.2b] Figure-8 target on hold: |B| = {np.max(b_t[hold]):.3f} T, |E| = {np.max(e_t[hold]):.2f} V/m")

# --- Task 2.2c: 피질 분포도 지표 ---
xs_2d = np.linspace(-120e-3, 120e-3, 121)
ys_2d = np.linspace(-120e-3, 120e-3, 121)
m_c = compute_cortical_metrics(circ_pts, circ_dl, DIDT_RISE, xs_2d, ys_2d, R_CIRC, l_circ)
m_f = compute_cortical_metrics(fig8_pts, fig8_dl, DIDT_RISE, xs_2d, ys_2d, 0.0, l_fig8)
for name, m in [("Circular", m_c), ("Figure-8", m_f)]:
    print(f"[Task 2.2c] {name:8s} | E_target = {m['E_target_Vm']:.2e} V/m, E_off = {m['E_off_target_Vm']:.2f} V/m, "
          f"S_1/2 = {m['S_half_cm2']:.2f} cm^2, d_1/2 = {m['d_half_from_scalp_mm']:.2f} mm, "
          f"V_req = {m['V_req_150_kV']:.2f} kV, Wm_req = {m['W_m_req_150_J']:.1f} J")


### Task 2.3 — 설계 변수 스윕과 트레이드오프 (14점)
8자형 코일의 기하 변수를 바꾸어 가며 깊이–초점성–전압/에너지 사이의 트레이드오프(trade-off)를 정량화함.

구현할 함수(`task2_tissue_induction.py`): `sweep_wing_radius`, `sweep_standoff`, `sweep_lateral_offset`


**검증 실행(check):** `task2_tissue_induction.py`에서 위 함수를 구현·저장한 뒤 아래 셀을 실행하면 검증 값이 출력됨.


In [ ]:
# --- Task 2.3: 설계 변수 스윕 ---
for row in sweep_wing_radius([25.0, 30.0, 35.0, 45.0, 55.0], xs_2d, ys_2d):
    print(f"[Task 2.3a] a = {row['radius_mm']:4.0f} mm | d_1/2 = {row['d_half_scalp_mm']:.2f} mm, "
          f"S_1/2 = {row['S_half_cm2']:.2f} cm^2, V_req = {row['V_req_150_kV']:.2f} kV, Wm_req = {row['W_m_req_150_J']:.1f} J")
for row in sweep_standoff([2.0, 5.0, 8.0, 10.0, 12.0, 15.0, 20.0, 25.0, 30.0], l_fig8):
    print(f"[Task 2.3b] h = {row['h_mm']:4.0f} mm | E_target = {row['E_target_Vm']:.2f} V/m, V_req = {row['V_req_150_kV']:.2f} kV")
for row in sweep_lateral_offset([0.0, 5.0, 10.0, 15.0, 20.0, 30.0, 40.0]):
    print(f"[Task 2.3c] dx = {row['dx_mm']:4.0f} mm | E_target = {row['E_target_Vm']:.2f} V/m")


## Task 3 — 대뇌 피질 뉴런 활성화 (20점)
제공되는 `neuron_model.py`의 `simulate_hh_neuron`, `find_electric_field_threshold`를 사용함. 유도 전기장과 뉴런의 결합은 handout의 Task 3 배경 설명을 참고함.


### Task 3.1 — 발화 선택성과 임계 구동 조건 (20점)
유도 전기장을 호지킨-헉슬리(Hodgkin–Huxley) 뉴런 모델에 입력하여 표적 뉴런만 선택적으로 발화하는지 확인하고, 발화에 필요한 최소 구동 조건을 구함.

구현할 함수(`task3_neuron.py`): `compute_E_eff_waveform`, `compute_fig8_thresholds`, `find_max_standoff`


**검증 실행(check):** `task3_neuron.py`에서 위 함수를 구현·저장한 뒤 아래 셀을 실행하면 검증 값이 출력됨.


In [ ]:
circ_pts, circ_dl = build_circular_coil()
fig8_pts, fig8_dl = build_figure8_coil()
l_fig8 = compute_inductance(fig8_pts, fig8_dl)
_, di_dt = trapezoidal_pulse_current_and_didt(T_S)


# --- Task 3.1a: 표적 vs. 비표적 뉴런 응답 ---
cases = {
    "Figure-8 Target    ": (fig8_pts, fig8_dl, R_TARGET),
    "Figure-8 Off-Target": (fig8_pts, fig8_dl, R_OFF_TARGET),
    "Circular Target    ": (circ_pts, circ_dl, R_TARGET),
    "Circular Off-Target": (circ_pts, circ_dl, R_OFF_TARGET),
}
e_fig8_target_peak = 0.0
for name, (pts, dl, r) in cases.items():
    e_eff = compute_E_eff_waveform(pts, dl, r, di_dt)
    hh = simulate_hh_neuron(T_MS, e_eff)
    print(f"[Task 3.1a] {name} | E_peak = {np.max(np.abs(e_eff)):8.2f} V/m, "
          f"Vm_max = {hh['peak_V_m']:+.2f} mV, AP = {hh['ap_triggered']}")
    if name.startswith("Figure-8 Target"):
        e_fig8_target_peak = float(np.max(np.abs(e_eff)))

# --- Task 3.1b/c: 임계 구동 조건 ---
e_th = find_electric_field_threshold(T_MS, di_dt / DIDT_RISE)
th = compute_fig8_thresholds(e_th, e_fig8_target_peak, l_fig8)
h_max = find_max_standoff(e_th)
print(f"[Task 3.1b] E_th = {e_th:.2f} V/m | I_peak_th = {th['I_peak_th_kA']:.3f} kA, "
      f"V_coil_th = {th['V_coil_th_kV']:.3f} kV, Wm_th = {th['W_m_th_J']:.2f} J")
print(f"[Task 3.1c] h_max = {h_max*1e3:.2f} mm (at I_peak = {I_PEAK_NOMINAL/1e3:.1f} kA)")


## Task 4 — 종합 고찰 (10점, 보고서만)
코드 구현 없이 보고서로 제출함.


## 보고서용 그림(figures)
위에서 구현한 함수를 사용해 보고서에 필요한 그림을 아래 셀부터 자유롭게 작성함.


In [ ]:
import matplotlib.pyplot as plt

# 예: z_line, e_mag, j_mag = compute_depth_profile(fig8_pts, fig8_dl, x_line=0.0)
